# CHỦ ĐỀ 1: ĐẦU TƯ VÀ TIẾP CẬN GIÁO DỤC

## 1. Phân tích cơ bản về dữ liệu

### 1.1. Giới thiệu tổng quan về dataset

Dataset được sử dụng trong bài phân tích này được thu thập từ **World Development Indicators (WDI)** do Ngân hàng Thế giới (World Bank) cung cấp. Dữ liệu bao gồm thông tin của **12 quốc gia** đa dạng về mức thu nhập và khu vực địa lý, trong giai đoạn từ **2015 đến 2022**.

Các quốc gia được phân tích: Brazil, France, Germany, India, Indonesia, Japan, Korea (Rep.), Singapore, South Africa, United Kingdom, United States, Viet Nam.

**Lý do chọn 12 quốc gia này:** nhằm tạo bối cảnh so sánh để **rút ra bài học và chiến lược phát triển giáo dục cho Việt Nam**, từ các nước phát triển và đang phát triển với mức đầu tư và tiếp cận giáo dục khác nhau.

### 1.2. Cấu trúc dữ liệu và các trường dữ liệu

| Cột | Mô tả |
|-----|--------|
| Country Name | Tên quốc gia |
| Country Code | Mã quốc gia (ISO 3-letter) |
| Series Name | Tên chỉ số |
| Series Code | Mã chỉ số |
| 2015 [YR2015] ... 2022 [YR2022] | Giá trị chỉ số theo từng năm |

Mỗi quốc gia có **3 chỉ số (Series):**

1. **`NY.GDP.PCAP.CD`** - GDP per capita (current US$): GDP bình quân đầu người theo giá hiện hành (USD).
2. **`SE.TER.ENRR`** - School enrollment, tertiary (% gross): Tỷ lệ nhập học bậc đại học/cao đẳng (% gross). Chỉ số này = [Số người nhập học đại học không phân biệt độ tuổi] / [Dân số độ tuổi đi học đại học chuẩn] × 100. Giá trị có thể >100% khi có nhiều sinh viên ngoài độ tuổi chuẩn.
3. **`SE.XPD.TOTL.GD.ZS`** - Government expenditure on education, total (% of GDP): Tổng chi tiêu công của chính phủ cho giáo dục tính theo % GDP.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Cài đặt tham số đồ họa
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid')

# 1. Đọc dữ liệu
DATA_PATH = 'dataset1.csv'
df_raw = pd.read_csv(DATA_PATH, encoding='utf-8-sig')

print(f"Kích thước bảng dữ liệu: {df_raw.shape[0]} dòng × {df_raw.shape[1]} cột")
print(f"Các cột: {list(df_raw.columns)}")
df_raw.head(6)

In [ ]:
# 2. Xử lý dữ liệu: chuyển từ Wide sang Long format (ĐỂ PHÙ HỢP CHO SEABORN MATPLOTLIB)
year_cols = [c for c in df_raw.columns if c.startswith('20') and c.endswith(']')]

df_long = df_raw.melt(
    id_vars=['Country Name', 'Country Code', 'Series Name', 'Series Code'],
    value_vars=year_cols,
    var_name='YearRaw', value_name='Value'
)

# Lấy ra con số năm (VD: "2015 [YR2015]" -> 2015)
df_long['Year'] = df_long['YearRaw'].str.extract(r'(\d{4})').astype(int)
df_long['Value'] = pd.to_numeric(df_long['Value'], errors='coerce')

# Lọc giai đoạn 2015-2022
df_long = df_long[(df_long['Year'] >= 2015) & (df_long['Year'] <= 2022)].copy()
df_long = df_long.sort_values(by=['Country Code', 'Series Code', 'Year']).reset_index(drop=True)

print(f"Số bản ghi sau chuyển đổi: {len(df_long)}")
print(f"Số giá trị thiếu (NaN): {df_long['Value'].isna().sum()}")
print(f"Các quốc gia: {sorted(df_long['Country Name'].unique())}")
print(f"Các chỉ số: {sorted(df_long['Series Code'].unique())}")

In [ ]:
# 3. Kiểm tra dữ liệu thiếu chi tiết
missing_detail = df_long[df_long['Value'].isna()][['Country Name', 'Series Code', 'Year']]
if len(missing_detail) > 0:
    print("Các ô dữ liệu bị thiếu (NaN):")
    print(missing_detail.to_string(index=False))
else:
    print("Không có dữ liệu bị thiếu.")

In [ ]:
# 4. Xử lý dữ liệu thiếu bằng nội suy tuyến tính trong nhóm (quốc gia, chỉ số)
df_clean = df_long.copy()
df_clean['Value'] = df_clean.groupby(['Country Code', 'Series Code'])['Value'].transform(
    lambda x: x.interpolate(method='linear').ffill().bfill()
)

print(f"Số giá trị thiếu sau xử lý: {df_clean['Value'].isna().sum()}")

In [ ]:
# 5. Thống kê cơ bản theo từng chỉ số
for code in sorted(df_clean['Series Code'].unique()):
    subset = df_clean[df_clean['Series Code'] == code]
    name = subset['Series Name'].iloc[0]
    print(f"\n--- {code}: {name} ---")
    print(subset['Value'].describe().to_string())

## Xử lý dữ liệu thiếu

Trong dataset, nhóm phát hiện một số giá trị bị thiếu (NaN) xuất hiện rải rác, ví dụ:

- Thiếu **chi tiêu giáo dục (% GDP)** ở France (2015–2016), United Kingdom (2022), Japan (2022), United States (2022)  
- Thiếu **tỷ lệ nhập học đại học** ở Singapore (2015) và Viet Nam (2018, 2020)

---

## Phương pháp xử lý

Nhóm sử dụng **_nội suy tuyến tính (linear interpolation)_** theo từng *(quốc gia, chỉ số)* để điền các giá trị thiếu.

---

## Lý do lựa chọn

- **_Dữ liệu dạng time-series_** → các chỉ số thay đổi khá **mượt theo thời gian**, nên có thể ước lượng dựa trên các năm lân cận  
- **_Tỷ lệ thiếu rất nhỏ_** (8/288) → không ảnh hưởng đáng kể đến xu hướng chung  
- **_Thiếu dữ liệu cục bộ_** → mỗi quốc gia/chỉ số chỉ thiếu ở **một vài năm riêng lẻ (tối đa 2 năm)**, không xuất hiện các khoảng thiếu dài liên tục
- **_Giữ lại dữ liệu_** → tránh phải xóa dòng, đảm bảo đủ dữ liệu để vẽ biểu đồ và phân tích
---

## Hạn chế

Nội suy chỉ là **_ước lượng_**, nên có thể sai lệch nhẹ so với thực tế. Tuy nhiên, do dữ liệu thiếu ít và không liên tục, nhóm đánh giá ảnh hưởng này là **không đáng kể**.

## 2. Xác định mục tiêu và lựa chọn trường dữ liệu

### 2.1. Mục tiêu 1: So sánh xu hướng và mức độ ưu tiên ngân sách quốc gia cho giáo dục

**Tiêu chí SMART:**
- **S (Specific):** Đánh giá nỗ lực đầu tư thông qua tỷ lệ chi tiêu công cho giáo dục (% GDP).
- **M (Measurable):** Dựa trên số liệu liên tục qua các năm của biến `SE.XPD.TOTL.GD.ZS`.
- **A (Achievable):** Hoàn toàn tính toán được nhờ dữ liệu sẵn có từ 12 quốc gia trong dataset.
- **R (Relevant):** Thể hiện bức tranh "Đầu tư giáo dục" của chính phủ.
- **T (Time-bound):** Từ năm 2015 đến 2022.

### Trường dữ liệu sử dụng

**`SE.XPD.TOTL.GD.ZS` - Government expenditure on education, total (% of GDP)**
- Chỉ số này đo lường tổng chi tiêu chung của chính phủ (bao gồm trung ương, địa phương) dành cho giáo dục, thể hiện dưới dạng tỷ lệ phần trăm so với GDP.
- **Phương pháp tính:** [Tổng chi tiêu công cho giáo dục ở tất cả các cấp học] / [GDP] × 100
- Chỉ số này phản ánh mức độ cam kết và ưu tiên của chính phủ đối với phát triển giáo dục so với quy mô kinh tế.
- **Lưu ý:** Chỉ tính **chi tiêu công** (ngân sách nhà nước), không bao gồm chi tiêu tư nhân (học phí do hộ gia đình đóng).

### Lý do lựa chọn metrics

Biến `SE.XPD.TOTL.GD.ZS` được chọn vì nó **chuẩn hóa theo quy mô kinh tế** (% GDP), cho phép so sánh công bằng giữa các quốc gia giàu và nghèo. Một nước có GDP nhỏ nhưng tỷ trọng chi tiêu giáo dục cao (như Viet Nam ~ 3-3.5%) cho thấy nỗ lực đầu tư lớn hơn tương đối so với một nước giàu chi ít (như Singapore ~ 2.5-3%).

###  Biểu đồ 1: Line Chart — Xu hướng chi tiêu giáo dục (% GDP) qua các năm

**Lý do chọn biểu đồ Line Chart:**
Line Chart là định dạng chuẩn mực nhất để theo dõi sự dịch chuyển của **time-series data**. Nó giúp ta dễ dàng so sánh:
- Độ dốc (tốc độ thay đổi) giữa các quốc gia
- Vị thế tương đối (quốc gia nào đầu tư mạnh hơn)
- Xu hướng chung: tăng, giảm, hoặc ổn định

Nhóm em sử dụng **highlight Min/Max** trên mỗi đường để làm nổi bật quốc gia có mức đầu tư cao nhất và thấp nhất tại mỗi thời điểm.

In [ ]:
# ====== BIỂU ĐỒ 1: LINE CHART — Chi tiêu giáo dục (% GDP) ======

# Lọc dữ liệu chi tiêu giáo dục
edu_df = df_clean[df_clean['Series Code'] == 'SE.XPD.TOTL.GD.ZS'].copy()

plt.figure(figsize=(16, 8))

# Vẽ line chart
palette = sns.color_palette('tab20', n_colors=len(edu_df['Country Name'].unique()))
ax = sns.lineplot(
    data=edu_df,
    x='Year', y='Value',
    hue='Country Name',
    marker='o', markersize=6,
    palette=palette,
    linewidth=2.5
)

# Highlight Min/Max theo từng năm
for year in edu_df['Year'].unique():
    year_data = edu_df[edu_df['Year'] == year]
    max_row = year_data.loc[year_data['Value'].idxmax()]
    min_row = year_data.loc[year_data['Value'].idxmin()]
    ax.annotate('▲', xy=(max_row['Year'], max_row['Value']),
                fontsize=10, color='red', ha='center', va='bottom')
    ax.annotate('▼', xy=(min_row['Year'], min_row['Value']),
                fontsize=10, color='blue', ha='center', va='top')

# Đường trung bình toàn bộ
overall_mean = edu_df.groupby('Year')['Value'].mean()
ax.plot(overall_mean.index, overall_mean.values, color='black', linestyle='--',
        linewidth=2, label='Trung bình 12 quốc gia', zorder=10)

plt.title('Xu hướng chi tiêu công cho giáo dục (% GDP) tại 12 quốc gia (2015–2022)',
          fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Năm', fontsize=12, fontweight='bold')
plt.ylabel('Chi tiêu giáo dục (% GDP)', fontsize=12, fontweight='bold')
plt.xticks(range(2015, 2023))
plt.ylim(0, 8)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=handles, labels=labels, title='Chú thích',
          bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)

plt.tight_layout()
plt.show()

**Phân tích biểu đồ Line Chart:**

### 1. Xu hướng chung:

Đầu tư giáo dục tính theo % GDP nhìn chung **tương đối ổn định** trong giai đoạn 2015–2022, với hầu hết các quốc gia chỉ **dao động nhẹ qua các năm**.  
--> **Indonesia là trường hợp ngoại lệ**, với mức giảm mạnh và rõ rệt so với các quốc gia còn lại. Điều này cho thấy chi tiêu giáo dục thường là một khoản **ngân sách ổn định**, ít biến động mạnh trong ngắn hạn, và những thay đổi lớn (như Indonesia) cần được xem xét thận trọng về bối cảnh hoặc cách đo lường.

### 2. Nhóm đầu tư cao (trên 4.5% GDP)

- **Brazil** duy trì mức chi tiêu cao (~5.5–6.3%) nhưng có xu hướng **giảm nhẹ**, cho thấy dù từng ưu tiên mạnh cho giáo dục, quốc gia này có thể đang chịu **áp lực tài khóa** hoặc phải phân bổ lại ngân sách cho các lĩnh vực khác.  

- **South Africa** không chỉ duy trì mức cao (~5.4–6.6%) mà còn có xu hướng **tăng dần**, cho thấy giáo dục đang trở thành một **trọng tâm chính sách dài hạn**, đặc biệt trong bối cảnh quốc gia này cần cải thiện chất lượng nguồn nhân lực.  

- **Korea (South Korea)** có xu hướng **tăng rõ rệt** theo thời gian, nổi bật hơn so với các quốc gia phát triển khác.  
  --> Điều này cho thấy quốc gia này vẫn đang trong giai đoạn **tăng cường đầu tư để nâng cao chất lượng nguồn nhân lực**, dù đã đạt mức phát triển cao.  

- **France, United Kingdom, Germany, United States** duy trì mức **ổn định quanh 4.5–5.5%**, phản ánh một đặc điểm chung của các nền kinh tế phát triển:  
  --> **chi tiêu giáo dục đã đạt trạng thái “cân bằng”**, nơi chính phủ duy trì mức đầu tư ổn định do hệ thống giáo dục đã tương đối hoàn thiện.

---

### 3. Nhóm đầu tư trung bình và thấp (dưới 4.5% GDP)

- **India** (~3.9–4.6%) duy trì mức trung bình thấp nhưng khá ổn định, cho thấy vì là quốc gia đông dân, việc mở rộng giáo dục có thể đang gặp hạn chế về nguồn lực ngân sách.  

- **Viet Nam** (~2.9–3.5%) có xu hướng giảm nhẹ, cho thấy tốc độ tăng chi tiêu giáo dục **chưa theo kịp tăng trưởng kinh tế**, hoặc nguồn lực đang được phân bổ sang các lĩnh vực phát triển khác.  

- **Japan** (~3.1–3.3%) duy trì mức thấp trong nhóm phát triển, cho thấy hệ thống giáo dục có thể **phụ thuộc nhiều hơn vào khu vực tư nhân**, thay vì chi tiêu công.  

- **Singapore** (~2.5–3%) có mức chi tiêu thấp nhất theo % GDP trong các nước đã phát triển. Tuy nhiên, do thu nhập bình quân đầu người cao, nên số tiền thực tế chi cho giáo dục vẫn lớn.  
--> Điều này cho thấy mỗi quốc gia có cách phân bổ ngân sách khác nhau, và % GDP không phải lúc nào cũng phản ánh đầy đủ mức đầu tư. 

- **Indonesia** giảm mạnh từ ~3.6% xuống ~1% và duy trì mức thấp, cho thấy sự thay đổi đáng kể trong cách phân bổ ngân sách hoặc phương pháp thống kê.  

---

### 4. Tác động COVID-19 (2020):

Một số quốc gia như **Germany, South Africa và United Kingdom** ghi nhận mức tăng nhẹ trong năm 2020. Tuy nhiên, sự gia tăng này **không nhất thiết phản ánh việc tăng chi tiêu thực tế**, mà có thể do **GDP giảm trong bối cảnh COVID-19**, khiến tỷ lệ chi tiêu (% GDP) tăng lên.


### Biểu đồ 2: Boxplot — Phân bổ chi tiêu giáo dục theo nhóm thu nhập

**Lý do chọn biểu đồ Boxplot:**
Boxplot giúp trực quan hóa **ranh giới phân bổ, độ lệch chuẩn, trung vị, và các mức đầu tư nằm ngoài chuẩn (outlier)** của từng nhóm thu nhập. Việc khảo sát theo nhóm thu nhập thay vì từng quốc gia riêng lẻ giúp ta đánh giá mức độ phân hóa trong đầu tư giáo dục, cũng như xem nền kinh tế ảnh hưởng thế nào đến sự phân bổ này.

In [ ]:
# ====== BIỂU ĐỒ 2: BOXPLOT — Phân bổ chi tiêu giáo dục ======

# Phân nhóm theo mức thu nhập (dựa trên GDP/capita trung bình)
gdp_df = df_clean[df_clean['Series Code'] == 'NY.GDP.PCAP.CD'].copy()
gdp_mean = gdp_df.groupby('Country Name')['Value'].mean().reset_index()
gdp_mean.columns = ['Country Name', 'GDP_Mean']

# Phân nhóm: Thu nhập cao (>30000), Trung bình cao (5000-30000), Trung bình thấp (<5000)
def classify_income(gdp):
    if gdp >= 30000:
        return 'Thu nhập cao (>$30K)'
    elif gdp >= 5000:
        return 'Thu nhập trung bình cao ($5K-$30K)'
    else:
        return 'Thu nhập trung bình thấp (<$5K)'

gdp_mean['Income_Group'] = gdp_mean['GDP_Mean'].apply(classify_income)

# Merge nhóm thu nhập vào dữ liệu giáo dục
edu_boxplot = edu_df.merge(gdp_mean[['Country Name', 'Income_Group']], on='Country Name', how='left')

plt.figure(figsize=(10, 6))

# Boxplot theo nhóm thu nhập
group_order = ['Thu nhập cao (>$30K)', 'Thu nhập trung bình cao ($5K-$30K)', 'Thu nhập trung bình thấp (<$5K)']
sns.boxplot(data=edu_boxplot, y='Income_Group', x='Value', order=group_order,
            palette='Set2', hue='Income_Group', legend=False)
plt.title('Phân bổ chi tiêu giáo dục (% GDP) theo nhóm thu nhập (2015–2022)',
                  fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Chi tiêu giáo dục (% GDP)', fontsize=12, fontweight='bold')
plt.ylabel('')

plt.tight_layout()
plt.show()

# In bảng phân nhóm
print("\nBảng phân nhóm thu nhập:")
for _, row in gdp_mean.sort_values('GDP_Mean', ascending=False).iterrows():
    print(f"  {row['Country Name']:20s} | GDP/capita trung bình: ${row['GDP_Mean']:,.0f} | Nhóm: {row['Income_Group']}")

**Phân tích biểu đồ Boxplot theo nhóm thu nhập:**

- Nhóm **thu nhập cao (>$30K)** có **mức chi tiêu trung vị cao và biên độ vừa phải**, cho thấy các nước giàu duy trì đầu tư ổn định nhưng vẫn có sự khác biệt nhất định giữa các quốc gia.

- Nhóm **thu nhập trung bình cao ($5K–$30K)** có **trung vị cao nhất và biên độ không quá lớn**, cho thấy mức đầu tư giáo dục **tương đối đồng đều giữa các quốc gia trong nhóm**.  
--> Điều này phản ánh xu hướng các nước đang phát triển mạnh **tăng cường đầu tư giáo dục nhằm hỗ trợ tăng trưởng và bứt phá kinh tế**.

- Nhóm **thu nhập trung bình thấp (<$5K)** có **biên độ lớn nhất**, cho thấy sự **phân hóa rõ rệt** giữa các quốc gia:  
  👉 Có nước đầu tư khá cao (ví dụ India), nhưng cũng có nước rất thấp (Indonesia). 

### Kết luận Mục tiêu 1

- **Nhóm quốc gia duy trì ngân sách giáo dục ổn định nhất:**  
  Nhóm **thu nhập cao**, đặc biệt là các nước Châu Âu như **France, Germany, United Kingdom**, duy trì mức chi tiêu ổn định quanh **4.5–5.5% GDP** trong suốt giai đoạn.  
  👉 Điều này phản ánh hệ thống giáo dục đã phát triển hoàn thiện, nơi chính phủ duy trì mức đầu tư ổn định thay vì tăng mạnh.

- **Quốc gia có tỷ trọng đầu tư cao nhất so với quy mô kinh tế:**  
  **South Africa** (~5.4–6.6%) và **Brazil** (~5.5–6.3%) là hai quốc gia nổi bật với mức chi tiêu cao nhất.  
  👉 Đây đều là các nước đang phát triển, cho thấy xu hướng **ưu tiên mạnh cho giáo dục nhằm cải thiện chất lượng nguồn nhân lực và thúc đẩy phát triển dài hạn**.

- **Sự khác biệt giữa các nhóm thu nhập:**  
  Boxplot cho thấy:
  - Nhóm **thu nhập trung bình cao** có mức chi tiêu **cao nhất và khá đồng đều** → đang trong giai đoạn **tăng tốc đầu tư để bứt phá**  
  - Nhóm **thu nhập cao** duy trì mức **ổn định** → phản ánh trạng thái “cân bằng”  
  - Nhóm **thu nhập trung bình thấp** có **mức phân hóa lớn nhất** → phụ thuộc vào năng lực tài chính và chính sách từng quốc gia  

- **Xu hướng chung:**  
  Đầu tư giáo dục (% GDP) nhìn chung **ổn định**, chỉ dao động nhẹ qua các năm ở đa số quốc gia.  
  👉 **Indonesia là ngoại lệ**, với mức giảm mạnh và duy trì ở mức thấp, cần được xem xét trong bối cảnh thay đổi chính sách hoặc phương pháp thống kê.  

  👉 Ngoài ra, năm **2020** ghi nhận sự tăng nhẹ ở một số quốc gia, nhưng chủ yếu do **GDP giảm vì COVID-19**, không phản ánh sự gia tăng chi tiêu thực tế.

- **Bài học rút ra cho Việt Nam:**  
  Việt Nam (~2.9–3.5%) vẫn nằm trong nhóm **chi tiêu trung bình thấp** và có xu hướng giảm nhẹ.  
  👉 Điều này cho thấy tỷ trọng đầu tư cho giáo dục **chưa theo kịp tốc độ tăng trưởng kinh tế**.

  👉 Để nâng cao chất lượng nguồn nhân lực và mở rộng tiếp cận giáo dục:
  - Cần **tăng dần tỷ trọng chi tiêu công cho giáo dục**  
  - Đồng thời có thể học hỏi các quốc gia như **Japan và Singapore** trong việc  
    **đa dạng hóa nguồn lực (kết hợp công – tư)** nhằm giảm áp lực ngân sách nhưng vẫn đảm bảo hiệu quả đầu tư

## 2.2. Mục tiêu 2: Đo lường tốc độ tăng trưởng mức độ tiếp cận giáo dục bậc cao

### Tiêu chí SMART:
- **S:** Tập trung vào khía cạnh "Tiếp cận giáo dục đại học/cao đẳng".
- **M:** Tính toán % tăng trưởng tuyệt đối/tương đối của biến `SE.TER.ENRR`.
- **A:** Có đủ số liệu đầu cuối hoặc có thể nội suy tuyến tính.
- **R:** Xác định xem quy mô giáo dục bậc cao đã mở rộng tới đâu.
- **T:** Đo lường mốc thời gian 2015 so với 2022.

### Trường dữ liệu sử dụng

**`SE.TER.ENRR` - School enrollment, tertiary (% gross)**
- Chỉ số này đo lường **tỷ lệ nhập học đại học/cao đẳng (% gross)**, tức là: [Số người nhập học bậc đại học không phân biệt độ tuổi] / [Dân số ở độ tuổi đi học đại học chuẩn] × 100.
- Giá trị **có thể vượt 100%** khi có nhiều sinh viên ngoài độ tuổi chuẩn (ví dụ: sinh viên lớn tuổi quay lại trường).
- Chỉ số phản ánh **mức độ phổ cập giáo dục bậc cao** của một quốc gia.

### Biểu đồ: Dumbbell Plot

**Lý do chọn biểu đồ Dumbbell Plot:**
Thay vì dùng Line Chart cho mục này (sẽ bị trùng lặp với Mục tiêu 1), **Dumbbell Plot** cực kỳ xuất sắc trong việc trực quan hóa **sự chênh lệch (gap)** giữa hai mốc thời gian Point A (2015) và Point B (2022). Người xem ngay lập tức thấy quốc gia nào có "đoạn thẳng" dài nhất = bứt phá mạnh nhất.

In [ ]:
# ====== BIỂU ĐỒ 3: DUMBBELL PLOT — Tăng trưởng nhập học đại học 2015 vs 2022 ======

ter_df = df_clean[df_clean['Series Code'] == 'SE.TER.ENRR'].copy()

# Lấy giá trị năm 2015 và 2022
ter_2015 = ter_df[ter_df['Year'] == 2015][['Country Name', 'Value']].rename(columns={'Value': 'Y2015'})
ter_2022 = ter_df[ter_df['Year'] == 2022][['Country Name', 'Value']].rename(columns={'Value': 'Y2022'})
dumbbell = ter_2015.merge(ter_2022, on='Country Name')
dumbbell['Change'] = dumbbell['Y2022'] - dumbbell['Y2015']
dumbbell['Change_pct'] = (dumbbell['Change'] / dumbbell['Y2015']) * 100
dumbbell = dumbbell.sort_values('Change', ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, 8))

# Vẽ thanh nối
for i, row in dumbbell.iterrows():
    color = '#2ecc71' if row['Change'] >= 0 else '#e74c3c'
    ax.plot([row['Y2015'], row['Y2022']], [i, i], color=color, linewidth=3, zorder=1)

# Vẽ điểm 2015 và 2022
ax.scatter(dumbbell['Y2015'], range(len(dumbbell)), color='#3498db', s=100, zorder=2, label='2015')
ax.scatter(dumbbell['Y2022'], range(len(dumbbell)), color='#e67e22', s=100, zorder=2, label='2022')

# Ghi chú thay đổi
for i, row in dumbbell.iterrows():
    sign = '+' if row['Change'] >= 0 else ''
    ax.annotate(f"{sign}{row['Change']:.1f}pp ({sign}{row['Change_pct']:.1f}%)",
                xy=(max(row['Y2015'], row['Y2022']) + 1.5, i),
                fontsize=9, va='center',
                color='#2ecc71' if row['Change'] >= 0 else '#e74c3c',
                fontweight='bold')

ax.set_yticks(range(len(dumbbell)))
ax.set_yticklabels(dumbbell['Country Name'], fontsize=11)
ax.set_xlabel('Tỷ lệ nhập học đại học (% gross)', fontsize=12, fontweight='bold')
ax.set_title('So sánh tỷ lệ nhập học đại học (% gross) – 2015 vs 2022\n(Dumbbell Plot)',
             fontsize=15, fontweight='bold', pad=15)
ax.legend(fontsize=11, loc='lower right')
ax.set_xlim(0, 130)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# In bảng chi tiết
print("\nTop 5 quốc gia có tốc độ bứt phá lớn nhất (tăng trưởng tuyệt đối):")
top5 = dumbbell.sort_values('Change', ascending=False).head(5)
for _, row in top5.iterrows():
    print(f"  {row['Country Name']:20s} | 2015: {row['Y2015']:.1f}% → 2022: {row['Y2022']:.1f}% | Thay đổi: +{row['Change']:.1f}pp (+{row['Change_pct']:.1f}%)")

## Phân tích biểu đồ Dumbbell Plot:

### 1. Top quốc gia bứt phá mạnh nhất:

- **United Kingdom** có mức tăng tuyệt đối lớn nhất (~+23pp), từ ~56% lên ~80%, cho thấy sự mở rộng mạnh mẽ của hệ thống giáo dục bậc cao.

- **Viet Nam** có mức tăng đáng kể (~+15pp, ~+50%), thuộc nhóm tăng nhanh nhất.  
  --> Điều này phản ánh nỗ lực mở rộng tiếp cận giáo dục bậc cao ở một quốc gia đang phát triển.

- **Brazil**, **Indonesia**, **India** cũng ghi nhận mức tăng tương đối lớn (~5–12pp), cho thấy xu hướng **mở rộng quy mô giáo dục bậc cao** ở các nước đang phát triển.


### 2. Nhóm quốc gia có tỷ lệ rất cao:

- **Korea (South Korea)** và **Singapore** duy trì mức rất cao (gần hoặc trên 90%) và vẫn tiếp tục tăng.  
  --> Điều này cho thấy giáo dục đại học đã **gần đạt mức phổ cập**.

### 3. Trường hợp đặc biệt:

- **United States** là quốc gia duy nhất có xu hướng **giảm** (~-9.5pp), từ ~89% xuống ~79%.  
  --> Điều này có thể phản ánh do chi phí học đại học quá cao, dẫn đến xu hướng tìm kiếm các con đường thay thế (nghề nghiệp, chứng chỉ ngắn hạn).


### 4. Xu hướng chung:

- Các quốc gia đang phát triển (Châu Á, Nam Mỹ) đang có tốc độ tăng trưởng nhanh hơn, cho thấy xu hướng **thu hẹp khoảng cách tiếp cận giáo dục đại học** so với các nước phát triển.

- Tuy nhiên, dù tốc độ tăng nhanh, tỷ lệ nhập học đại học hiện tại vẫn thấp hơn đáng kể so với các quốc gia đã đạt mức phổ cập cao.

### 🇻🇳 Insight cho Việt Nam:

Việt Nam là một trong những quốc gia có **tốc độ tăng trưởng nhanh về tỷ lệ nhập học đại học**, cho thấy chính sách mở rộng giáo dục đã đạt hiệu quả rõ rệt.

Tuy nhiên, mức tuyệt đối (~45%) vẫn còn thấp so với các nước phát triển (70–100%).  
--> Điều này cho thấy Việt Nam đang ở **giai đoạn mở rộng quy mô**, chưa đạt mức phổ cập.

--> Trong thời gian tới, Việt Nam cần:
- Tiếp tục **mở rộng cơ hội tiếp cận giáo dục đại học**
- Đồng thời chú trọng **nâng cao chất lượng đào tạo**, tránh chạy theo số lượng

## 2.3. Mục tiêu 3: Phân tích mối quan hệ tương quan tuyến tính giữa đầu tư giáo dục và tiếp cận giáo dục

### Tiêu chí SMART:
- **S:** Trả lời câu hỏi: "Cứ chi nhiều tiền (% GDP) thì tỷ lệ người dân được đi học đại học có tăng thuận chiều không?"
- **M:** Sử dụng hệ số tương quan Pearson $r$ giữa 2 biến `SE.XPD.TOTL.GD.ZS` và `SE.TER.ENRR`.
- **A:** Tính toán dễ dàng thông qua Pandas và NumPy.
- **R:** Móc nối trực tiếp 2 vế của chủ đề "Đầu tư" và "Tiếp cận".
- **T:** Tổng hợp trong rổ dữ liệu giai đoạn 2015–2022.

### Biểu đồ: Biểu đồ kết hợp cột và đường (Dual-axis Combo Chart)

**Lý do chọn biểu đồ Combo Chart:**
Việc sử dụng biểu đồ kết hợp (Cột thể hiện chi tiêu giáo dục, Đường thể hiện tỷ lệ nhập học) trên **hai trục tung (Dual-axis)** tính theo mức trung bình giúp chúng ta dễ dàng so sánh một cách trực quan độ lớn của hai biến số. Nếu hai biến có tương quan thuận, đường (Line) sẽ có xu hướng nhấp nhô đồng điệu với độ cao của các cột (Bar).

In [ ]:
# ====== BIỂU ĐỒ 4: COMBO CHART — Tương quan Đầu tư vs Tiếp cận ======

# Chuẩn bị dữ liệu: pivot ra 3 biến cho mỗi (quốc gia, năm)
pivot_df = df_clean.pivot_table(
    index=['Country Name', 'Country Code', 'Year'],
    columns='Series Code',
    values='Value'
).reset_index()
pivot_df.columns.name = None

# Đổi tên cột cho dễ đọc
pivot_df = pivot_df.rename(columns={
    'SE.XPD.TOTL.GD.ZS': 'Edu_Expenditure',
    'SE.TER.ENRR': 'Tertiary_Enrollment',
    'NY.GDP.PCAP.CD': 'GDP_per_capita'
})

# Loại bỏ hàng thiếu dữ liệu
scatter_df = pivot_df.dropna(subset=['Edu_Expenditure', 'Tertiary_Enrollment', 'GDP_per_capita']).copy()

# Tính giá trị trung bình giai đoạn 2015-2022 cho mỗi quốc gia để vẽ Combo chart
avg_df = scatter_df.groupby('Country Name')[['Edu_Expenditure', 'Tertiary_Enrollment']].mean().reset_index()
# Sắp xếp theo chi tiêu giáo dục trung bình giảm dần
avg_df = avg_df.sort_values('Edu_Expenditure', ascending=False)

# Tính hệ số tương quan Pearson tổng thể để so sánh (đáp ứng tiêu chí SMART)
r_value, p_value = stats.pearsonr(scatter_df['Edu_Expenditure'], scatter_df['Tertiary_Enrollment'])

fig, ax1 = plt.subplots(figsize=(14, 8))

# TRỤC 1 (Trái): Bar chart cho Chi tiêu giáo dục (% GDP)
color_bar = '#3498db'
bars = ax1.bar(avg_df['Country Name'], avg_df['Edu_Expenditure'], color=color_bar, alpha=0.7, width=0.6, label='Khối lượng đầu tư (Chi tiêu % GDP)')
ax1.set_xlabel('Quốc gia', fontsize=13, fontweight='bold')
ax1.set_ylabel('Chi tiêu giáo dục trung bình (% GDP)', color=color_bar, fontsize=13, fontweight='bold')
ax1.tick_params(axis='y', labelcolor=color_bar)
ax1.set_ylim(0, avg_df['Edu_Expenditure'].max() * 1.25)
plt.xticks(rotation=45, ha='right', fontsize=11)

# Ghi chú giá trị trên cột
for bar in bars:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, yval + 0.1, f'{yval:.1f}%', ha='center', va='bottom', fontsize=9, color=color_bar, fontweight='bold')

# TRỤC 2 (Phải): Line chart cho Tỷ lệ nhập học đại học (%)
ax2 = ax1.twinx()
color_line = '#e74c3c'
line = ax2.plot(avg_df['Country Name'], avg_df['Tertiary_Enrollment'], color=color_line, marker='o', 
                linewidth=3, markersize=10, label='Mức độ tiếp cận (Tỷ lệ nhập học)')
ax2.set_ylabel('Tỷ lệ nhập học đại học trung bình (% gross)', color=color_line, fontsize=13, fontweight='bold')
ax2.tick_params(axis='y', labelcolor=color_line)
ax2.set_ylim(0, max(130, avg_df['Tertiary_Enrollment'].max() * 1.15))

# Ghi chú giá trị trên đường
for i, txt in enumerate(avg_df['Tertiary_Enrollment']):
    # Dùng offset linh hoạt để text không đè lên cột
    ax2.annotate(f'{txt:.0f}%', (avg_df['Country Name'].iloc[i], txt), textcoords="offset points", xytext=(0,12), ha='center', fontsize=10, color=color_line, fontweight='bold')

# Hiển thị biểu đồ kết luận tương quan phía trên
ax1.text(0.02, 0.96,
        f'Hệ số Pearson r = {r_value:.4f} (p-value = {p_value:.4f})\nKết luận: Tương quan yếu, chi tiêu % GDP cao chưa chắc tỷ lệ nhập học cao.',
        transform=ax1.transAxes, fontsize=12, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.title('Đánh giá tương quan giữa Đầu tư giáo dục và Mức độ tiếp cận Đại học\n(Trung bình giai đoạn 2015-2022)',
          fontsize=16, fontweight='bold', pad=20)

# Gộp legend
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper right', fontsize=11)

plt.tight_layout()
plt.show()

print(f"\n{'='*60}")
print(f"KẾT QUẢ PHÂN TÍCH TƯƠNG QUAN (Toàn bộ dữ liệu panel)")
print(f"{'='*60}")
print(f"Hệ số tương quan Pearson r = {r_value:.4f}")
print(f"p-value = {p_value:.4f}")
print(f"Số quan sát tổng cộng: {len(scatter_df)}")
if abs(r_value) < 0.3:
    print(f"→ Kết luận thống kê: Tương quan YẾU (|r| < 0.3)")
elif abs(r_value) < 0.5:
    print(f"→ Kết luận thống kê: Tương quan TRUNG BÌNH (0.3 ≤ |r| < 0.5)")
elif abs(r_value) < 0.7:
    print(f"→ Kết luận thống kê: Tương quan KHÁ (0.5 ≤ |r| < 0.7)")
else:
    print(f"→ Kết luận thống kê: Tương quan MẠNH (|r| ≥ 0.7)")

**Phân tích biểu đồ Combo Chart:**

1. **Về hình dáng đồ thị (Đường vs. Cột):**
   - Sự không đồng điệu thể hiện rất rõ ràng: Độ cao của cột Xanh (Chi tiêu % GDP) giảm dần từ trái qua phải, nhưng đường Đỏ (Tỷ lệ nhập học) lại nhấp nhô lộn xộn, thậm chí đạt đỉnh ở những quốc gia có cột rất thấp. Điều này cho thấy **chi tiêu nhiều % GDP chưa chắc đã mang lại tỷ lệ người dân đi học đại học cao hơn.**

2. **Chỉ ra các Outliers (Ngoại lệ):**
   - **Korea, Rep.** và **Singapore**: Cột chi tiêu bên trục trái khá thấp (chỉ khoảng 2.5–5% GDP) nhưng đường nối tỷ lệ nhập học lại vươn lên mốc rất cao (>90%). Lý do: Khu vực tư nhân (trường tư thục, học phí từ gia đình) gánh phần lớn chi phí giáo dục của hai quốc gia này. Đồng thời, nền tảng GDP tuyệt đối của họ rất lớn nên tỷ lệ % dù thấp vẫn là một số tiền khổng lồ chuyên biệt rải xuống.
   - **South Africa** và **Brazil**: Cột chi tiêu bên trục trái cao nhất (5.5–6.5% GDP) nhưng điểm thuộc đường màu đỏ lại lặn xuống dưới mốc ~20-60%. Điều này cho thấy tính hiệu quả trong việc phân bổ ngân sách cho giáo dục bậc cao chưa đạt kỳ vọng, hoặc họ phải ưu tiên chi ngân sách để xoá mù chữ và giáo dục cơ sở trước khi lo đến bậc đại học.

3. **Về hệ số tương quan Pearson:**
   - Hệ số $r$ dao động gần 0, khẳng định về mặt thống kê rằng tương quan tuyến tính là rất **yếu**.

4. **Suy luận:**
   - Việc "Đầu tư (% GDP)" không dịch chuyển tương đương với "Tiếp cận (Tỷ lệ nhập học)" chứng minh rằng hệ thống giáo dục một quốc gia chịu chi phối bởi rất nhiều biến số khác: Hệ thống sinh thái các trường tư thục, Thu nhập bình quân đầu người thực tế (giá trị tuyệt đối của ngân sách), định hướng nghề nghiệp, và Đặc điểm dân số học.


# 3. Tổng kết phân tích đầu tư và tiếp cận giáo dục (2015–2022)

## 3.1. Đầu tư giáo dục (% GDP)

- **Xu hướng chung:** Hầu hết các quốc gia duy trì **chi tiêu ổn định hoặc giảm nhẹ** trong giai đoạn 2015–2022. Chi tiêu giáo dục thường là một **ngân sách ổn định**, ít biến động trong ngắn hạn.  
- **Nhóm chi tiêu cao (>5% GDP):** South Africa tăng dần, Brazil giảm nhẹ → các nước đang phát triển ưu tiên giáo dục nhưng phải cân nhắc áp lực ngân sách.  
- **Nhóm chi tiêu trung bình/thấp:** Việt Nam (~2.9–3.5%) và một số nước châu Á thấp hơn, có xu hướng giảm nhẹ → tỷ trọng đầu tư công **chưa theo kịp tăng trưởng kinh tế**, cần cân nhắc tăng dần và kết hợp nguồn lực tư nhân.  
- **Tác động COVID-19:** Một số nước ghi nhận tăng nhẹ tỷ lệ chi tiêu/GDP, chủ yếu do **GDP giảm**, không phản ánh chi tiêu thực tế tăng.  

**🇻🇳 Insight Việt Nam:**  
- Tỷ trọng chi tiêu trung bình thấp, nên cần **tăng đầu tư công dần** và **đa dạng hóa nguồn lực (công – tư)** để nâng cao chất lượng giáo dục mà không tạo áp lực ngân sách.  

---

## 3.2. Mức độ tiếp cận giáo dục bậc cao

- **Xu hướng tăng nhanh:** Các nước đang phát triển như Việt Nam, Brazil, Indonesia chứng kiến mức tăng **15pp–23pp**, phản ánh nỗ lực mở rộng hệ thống đại học.  
- **Nhóm phổ cập cao:** Korea, Singapore (>90%), tiếp tục tăng → hệ thống gần đạt mức phổ cập, phần lớn chi phí gánh bởi khu vực tư nhân.  
- **Trường hợp đặc biệt:** Mỹ giảm (~-9.5pp) → chi phí học đại học cao, dẫn đến tìm kiếm các con đường thay thế.  

**🇻🇳 Insight Việt Nam:**  
- Tăng nhanh (~+15pp, hiện ~45%) → đang trong **giai đoạn mở rộng quy mô**, chưa phổ cập.  
- Cần tiếp tục **tăng cơ hội tiếp cận** và **nâng cao chất lượng đào tạo**, tránh chạy theo số lượng.  

---

## 3.3. Tương quan chi tiêu – tiếp cận (Combo chart)

- **Quan sát trực quan:** Cột chi tiêu (% GDP) không tương ứng trực tiếp với đường tỷ lệ nhập học → **chi tiêu nhiều chưa chắc mang lại tiếp cận cao**.  
- **Outliers:**  
  - Korea, Singapore: chi tiêu % thấp nhưng tỷ lệ nhập học rất cao → GDP tuyệt đối lớn và sự tham gia mạnh của khu vực tư nhân.  
  - South Africa, Brazil: chi tiêu cao nhưng tỷ lệ nhập học đại học chưa tương xứng → phân bổ ngân sách chưa tối ưu hoặc ưu tiên giáo dục cơ sở.  
- **Hệ số Pearson r ≈ 0.09, p ≈ 0.38:** tương quan tuyến tính **rất yếu và không đáng tin**, khẳng định chi tiêu % GDP **không phải yếu tố quyết định duy nhất**.  

**🇻🇳 Insight Việt Nam:**  
- Tăng chi tiêu % GDP chưa chắc đảm bảo tỷ lệ nhập học cao → cần cân nhắc:  
  - **Phân bổ ngân sách hợp lý giữa các cấp học**  
  - **Khuyến khích giáo dục tư nhân và học phí phù hợp**  
  - **Đầu tư đồng bộ chất lượng và tiếp cận**, thay vì chỉ tăng ngân sách chung  

---

## 3.4. Kết luận tổng thể

1. **Đầu tư giáo dục (% GDP):** ổn định hoặc giảm nhẹ; Việt Nam thấp, cần tăng dần và kết hợp tư nhân.  
2. **Mức độ tiếp cận đại học:** tăng nhanh ở các nước đang phát triển; Việt Nam tiến bộ rõ rệt nhưng chưa phổ cập.  
3. **Tương quan chi tiêu – tiếp cận:** yếu, không tuyến tính; chi tiêu nhiều không đảm bảo tỷ lệ nhập học cao.  